In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("../data/data.csv")
df.drop("no_referensi", axis=1, inplace=True)
df.drop("kondisi_kartu", axis=1, inplace=True)

In [4]:
df.head()

,kolek,bucket,remarks,tgl_approval,net_salary,card_product_name,age,status_deviasi,plafon,customer_gender,marital_status
0,1,0).CURRENT,TELEMARKETING,2020/09/02,"24,460,337.00",VISA PLATINUM,49,NON DEVIASI,"20,000,000.00",PEREMPUAN,MENIKAH
1,1,0).CURRENT,TELEMARKETING,2020/03/11,"45,000,000.00",VISA PLATINUM,37,NON DEVIASI,"20,000,000.00",LAKI-LAKI,MENIKAH
2,1,0).CURRENT,BRANCH,2020/04/01,"1,630,823,468.00",VISA PLATINUM,61,FINANCIAL,"50,000,000.00",PEREMPUAN,MENIKAH
3,1,0).CURRENT,TELEMARKETING,2020/12/11,"21,528,243.00",VISA PLATINUM,39,FINANCIAL,"20,000,000.00",LAKI-LAKI,MENIKAH
4,1,0).CURRENT,BRANCH,2020/10/27,"62,605,667.00",VISA PLATINUM,40,NON DEVIASI,"20,000,000.00",LAKI-LAKI,MENIKAH


In [5]:
df["tgl_approval"] = pd.to_datetime(df["tgl_approval"])

In [6]:
df["target"] = df["kolek"].apply(lambda x: 0 if x == 1 else 1)
df.drop("kolek", axis=1, inplace=True)

In [7]:
df = df.sort_values("tgl_approval", ignore_index=True)

In [8]:
df.head()

,bucket,remarks,tgl_approval,net_salary,card_product_name,age,status_deviasi,plafon,customer_gender,marital_status,target
0,0).CURRENT,BRANCH,2020-01-02,"24,253,100.00",VISA GOLD,29,NON DEVIASI,"12,000,000.00",LAKI-LAKI,LAJANG,0
1,0).CURRENT,BRANCH,2020-01-02,"30,289,427.00",VISA GOLD,25,NON DEVIASI,"6,000,000.00",LAKI-LAKI,LAJANG,0
2,0).CURRENT,BRANCH,2020-01-02,"7,861,140.00",VISA GOLD,37,FINANCIAL,"8,000,000.00",LAKI-LAKI,MENIKAH,0
3,0).CURRENT,BRANCH,2020-01-02,"2,602,470,175.00",VISA PLATINUM,62,NON DEVIASI,"49,000,000.00",LAKI-LAKI,MENIKAH,0
4,0).CURRENT,BRANCH,2020-01-02,"19,120,480.00",VISA GOLD,45,NON DEVIASI,"6,000,000.00",PEREMPUAN,MENIKAH,0


In [9]:
split_idx = int(len(df) * 0.8)

train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

train.drop("tgl_approval", axis=1, inplace=True)
test.drop("tgl_approval", axis=1, inplace=True)

train.to_csv("../data/train.csv", index=False)
test.to_csv("../data/test.csv", index=False)

print("Split done")
print(f"Train: {train.shape}, Test: {test.shape}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_36216\399544164.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train.drop("tgl_approval", axis=1, inplace=True)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_36216\399544164.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test.drop("tgl_approval", axis=1, inplace=True)


Split done
Train: (54564, 10), Test: (13642, 10)


In [11]:
# src/train.py
import joblib
from optbinning import BinningProcess, Scorecard
from sklearn.linear_model import LogisticRegression

In [12]:
TRAIN_PATH = "../data/train.csv"
MODEL_PATH = "../models/scorecard.pkl"

TARGET = "target"

df = pd.read_csv(TRAIN_PATH)

y = df[TARGET]
X = df.drop(columns=[TARGET])

categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object", "string"]).columns.tolist()

# define binning
binning_process = BinningProcess(
    variable_names=X.columns.tolist(),
    categorical_variables=categorical_cols
)

# model
estimator = LogisticRegression(max_iter=1000)

# scorecard
scorecard = Scorecard(
    binning_process=binning_process,
    estimator=estimator
)

scorecard.fit(X, y)

joblib.dump(scorecard, MODEL_PATH)

print("Model trained & saved")

Model trained & saved


In [13]:
from sklearn.metrics import roc_auc_score, classification_report

TEST_PATH = "../data/test.csv"
MODEL_PATH = "../models/scorecard.pkl"

TARGET = "target"

def main():
    df = pd.read_csv(TEST_PATH)
    y_true = df[TARGET]
    X = df.drop(columns=[TARGET])

    model = joblib.load(MODEL_PATH)

    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_true, y_proba)

    print("AUC:", auc)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

if __name__ == "__main__":
    main()

AUC: 0.5821166250899874

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12276
           1       0.13      0.00      0.01      1366

    accuracy                           0.90     13642
   macro avg       0.52      0.50      0.48     13642
weighted avg       0.82      0.90      0.85     13642

